In [1]:
%pip install pandas nltk
import nltk
nltk.download('punkt')
nltk.download('words')
nltk.download('punkt_tab')


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\danjo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\danjo\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\danjo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
import pandas as pd
import re

def load_tagalog_dictionary_from_csv(filename):
    """Load Tagalog dictionary from local CSV file with proper encoding handling"""
    try:
        df_dict = pd.read_csv(filename, header=None, names=['word'], encoding='utf-8')
    except UnicodeDecodeError:
        # Try fallback encoding if UTF-8 fails
        df_dict = pd.read_csv(filename, header=None, names=['word'], encoding='latin1')
    
    tagalog_words = set(df_dict['word'].dropna().astype(str).str.strip().str.lower())
    return tagalog_words

# Load dictionary and data with explicit paths
try:
    tagalog_words = load_tagalog_dictionary_from_csv("tagalog_dict.csv")
    df = pd.read_csv("tagalog_reddit_data.csv", encoding='utf-8')
except FileNotFoundError as e:
    print(f"File not found: {e}")
    exit()

# Verify dataframe structure
print("Columns in dataset:", df.columns.tolist())

# Common English phrases to filter
english_phrases = [
    'are you affiliated to certain religous group',
    'fuck off',
    'i cant even believe',
    'as someone from',
    'i cant even',
    'this conversation',
]

def should_filter_out(text):
    """Improved filtering with empty string handling"""
    if pd.isna(text) or str(text).strip() == '':
        return True
        
    text_lower = str(text).lower()
    
    # Check for banned phrases
    if any(phrase in text_lower for phrase in english_phrases):
        return True
    
    # Tokenize and count Tagalog words
    words = re.findall(r'\b\w+\b', text_lower)
    if not words:
        return True
        
    tagalog_count = sum(1 for word in words if word in tagalog_words)
    return (tagalog_count / len(words) < 0.2) and (tagalog_count < 3)

def preprocess_tagalog(text):
    """Enhanced preprocessing with edge case handling"""
    if pd.isna(text):
        return ""
        
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>', '', text)  # Remove HTML
    text = re.sub(r'[^\w\sñÑ]', ' ', text)  # Keep alphanumeric + ñ/Ñ
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Main processing
TEXT_COLUMN = 'Text'  # Make case-sensitive column name explicit

if TEXT_COLUMN in df.columns:
    # Filtering
    df['should_filter'] = df[TEXT_COLUMN].apply(should_filter_out)
    df_filtered = df[~df['should_filter']].copy()
    
    print(f"Filtered out {len(df)-len(df_filtered)}/{len(df)} entries")
    
    # Preprocessing
    df_filtered['processed_text'] = df_filtered[TEXT_COLUMN].apply(preprocess_tagalog)
    df_filtered['emotion'] = ""
    df_filtered.drop(columns=['should_filter'], inplace=True)
    
    # Save and show results
    output_filename = 'preprocessed_tagalog_data.csv'
    df_filtered.to_csv(output_filename, index=False)
    
    print(f"\nSaved to {output_filename}")
    print(f"Final dataset size: {len(df_filtered)}")
    
    # Show samples
    print("\nSample processed texts:")
    for i in range(min(3, len(df_filtered))):
        print(f"Original: {df_filtered[TEXT_COLUMN].iloc[i]}")
        print(f"Processed: {df_filtered['processed_text'].iloc[i]}")
        print("-" * 80)
else:
    print(f"Error: '{TEXT_COLUMN}' column missing. Available columns: {df.columns.tolist()}")

Columns in dataset: ['#', 'Text']
Filtered out 1109/1109 entries

Saved to preprocessed_tagalog_data.csv
Final dataset size: 0

Sample processed texts:
